# Shows Unified - Camada Gold

**Objetivo:** Unificar dados de todas as fontes em uma única tabela consolidada

---

## Tabelas de Origem (Silver)

* `skztour.02_silver.clean_setlist_fm` (~20 shows)
* `skztour.02_silver.clean_wikipedia_dominate` (~57 shows)
* `skztour.02_silver.clean_maniac_5star` (~52 shows)

## Tabela de Destino (Gold)

* `skztour.03_gold.shows_unified` (~130 shows)

---

## Schema Unificado

```
show_id: string          # ID único do show
source: string           # setlist_fm | wikipedia_maniac_5star | wikipedia_dominate
event_date: date         # Data do show (yyyy-MM-dd)
event_year: int          # Ano extraído
artist_name: string      # "Stray Kids"
venue_name: string       # Nome do local
venue_city: string       # Cidade
country: string          # País
attendance: int          # Público (gerado sinteticamente onde NULL)
source_url: string       # URL da fonte original
```

---

## Operação

**UNION ALL** das 3 tabelas silver (todas já têm o mesmo schema padronizado) 

In [0]:
from pyspark.sql import functions as F

# Ler as 3 tabelas limpas da camada silver
df_setlist = spark.table("skztour.02_silver.setlist_fm")

# Wikipedia Dominate precisa de colunas adicionais para match com as outras
df_dominate_raw = spark.table("skztour.02_silver.wikipedia_dominate")
df_dominate = df_dominate_raw \
    .withColumn("event_year", F.year("event_date")) \
    .withColumn("attendance", F.expr("try_cast(regexp_replace(attendance, ',', '') as int)"))

df_maniac = spark.table("skztour.02_silver.clean_maniac_5star")

# Padronizar attendance como integer em todas as tabelas
# Remove vírgulas e converte para integer (try_cast para lidar com valores inválidos)
df_setlist = df_setlist.withColumn("attendance", 
    F.expr("try_cast(regexp_replace(attendance, ',', '') as int)"))

# Selecionar apenas as colunas necessárias (removendo sets_json e info)
columns_to_keep = ["show_id", "source", "event_date", "event_year", "artist_name", 
                   "venue_name", "venue_city", "country", "attendance", "source_url"]

df_setlist = df_setlist.select(*columns_to_keep)
df_dominate = df_dominate.select(*columns_to_keep)
df_maniac = df_maniac.select(*columns_to_keep)

print("Contagem por fonte:")
print(f"  - Setlist.fm: {df_setlist.count()} shows")
print(f"  - Wikipedia Dominate: {df_dominate.count()} shows")
print(f"  - Wikipedia Maniac 5-STAR: {df_maniac.count()} shows")

# Unir todas as tabelas (já têm o mesmo schema!)
df_unified = df_setlist \
    .unionByName(df_dominate) \
    .unionByName(df_maniac) \
    .orderBy("event_date")

print(f"\nTotal shows unified: {df_unified.count()}")

# Validações
print("\nShows por ano:")
df_unified.groupBy("event_year").count().orderBy("event_year").show()

print("\nShows por país (Top 10):")
df_unified.groupBy("country").count().orderBy(F.desc("count")).show(10)

print("\nShows por fonte:")
df_unified.groupBy("source").count().orderBy("source").show()

Contagem por fonte:
  - Setlist.fm: 20 shows
  - Wikipedia Dominate: 57 shows
  - Wikipedia Maniac 5-STAR: 52 shows

Total shows unified: 129

📅 Shows por ano:
+----------+-----+
|event_year|count|
+----------+-----+
|      2022|   23|
|      2023|   29|
|      2025|   70|
|      2026|    7|
+----------+-----+


🌍 Shows por país (Top 10):
+-------------+-----+
|      country|count|
+-------------+-----+
|        Japan|   32|
|United States|   29|
|  South Korea|   24|
|    Australia|    6|
|        China|    4|
|       Taiwan|    3|
|  Philippines|    3|
|       France|    3|
|     Thailand|    3|
|    Indonesia|    3|
+-------------+-----+
only showing top 10 rows

📍 Shows por fonte:
+--------------------+-----+
|              source|count|
+--------------------+-----+
|          setlist_fm|   20|
|           wikipedia|   57|
|wikipedia_maniac_...|   52|
+--------------------+-----+



In [0]:
# ============================================
# GERAÇÃO DE DADOS SINTÉTICOS DE PÚBLICO
# ============================================

import random
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("🎭 Gerando dados de público baseados em fatores realistas...\n")

# Definir capacidades base por tipo de venue (baseado em palavras-chave)
venue_capacities = {
    # Stadiums (50k-90k)
    "stadium": (50000, 90000),
    "dome": (40000, 70000),
    
    # Arenas (10k-20k)
    "arena": (10000, 20000),
    "center": (8000, 18000),
    "hall": (5000, 12000),
    
    # Theaters/Clubs (1k-5k)
    "theater": (1000, 5000),
    "club": (500, 3000),
}

# Multiplicadores por país (baseado em tamanho de mercado K-pop)
country_multipliers = {
    "South Korea": 1.3,      # Mercado doméstico
    "Japan": 1.4,            # Mercado gigante
    "United States": 1.2,    # Mercado crescente
    "Thailand": 1.1,
    "Philippines": 1.1,
    "Singapore": 1.0,
    "Indonesia": 1.1,
    "Australia": 1.0,
    "default": 0.9           # Outros países
}

# Multiplicador por ano (popularidade crescente)
year_multipliers = {
    2018: 0.5,
    2019: 0.7,
    2020: 0.8,  # COVID impact
    2021: 0.6,  # COVID impact  
    2022: 1.0,
    2023: 1.3,
    2024: 1.5,
}

def classify_venue(venue_name):
    """Classifica o tipo de venue baseado no nome"""
    if not venue_name:
        return "arena"  # default
    
    venue_lower = venue_name.lower()
    
    # Check em ordem de especificidade
    if "stadium" in venue_lower or "dome" in venue_lower:
        return "dome" if "dome" in venue_lower else "stadium"
    elif "arena" in venue_lower:
        return "arena"
    elif "center" in venue_lower or "centre" in venue_lower:
        return "center"
    elif "hall" in venue_lower:
        return "hall"
    elif "theater" in venue_lower or "theatre" in venue_lower:
        return "theater"
    elif "club" in venue_lower:
        return "club"
    else:
        return "arena"  # default para venues desconhecidos

def generate_attendance(venue_name, country, year):
    """Gera um número de público realista"""
    
    # Classificar venue
    venue_type = classify_venue(venue_name)
    base_min, base_max = venue_capacities.get(venue_type, (8000, 15000))
    
    # Aplicar multiplicadores
    country_mult = country_multipliers.get(country, country_multipliers["default"])
    year_mult = year_multipliers.get(year, 1.0)
    
    # Calcular range ajustado
    adjusted_min = int(base_min * country_mult * year_mult)
    adjusted_max = int(base_max * country_mult * year_mult)
    
    # Gerar número aleatório no range (80%-100% da capacidade para simular sold out/near sold out)
    attendance = random.randint(int(adjusted_min * 0.8), adjusted_max)
    
    # Arredondar para centenas (mais realista)
    return (attendance // 100) * 100

# Registrar UDF
generate_attendance_udf = F.udf(generate_attendance, "int")

# Aplicar geração de attendance apenas onde é NULL
df_with_attendance = df_unified.withColumn(
    "attendance",
    F.when(
        F.col("attendance").isNull(),
        generate_attendance_udf(F.col("venue_name"), F.col("country"), F.col("event_year"))
    ).otherwise(F.col("attendance"))
)

# Estatísticas
print("📊 Estatísticas de Público:\n")
print(f"Total de shows: {df_with_attendance.count()}")
print(f"Shows com público: {df_with_attendance.filter(F.col('attendance').isNotNull()).count()}")
print(f"Shows sem público: {df_with_attendance.filter(F.col('attendance').isNull()).count()}")

print("\n🎯 Público por País (Top 10):")
df_with_attendance.groupBy("country") \
    .agg(
        F.count("*").alias("shows"),
        F.sum("attendance").alias("total_attendance"),
        F.round(F.avg("attendance"), 0).alias("avg_attendance")
    ) \
    .orderBy(F.desc("total_attendance")) \
    .show(10, truncate=False)

print("\n📅 Público por Ano:")
df_with_attendance.groupBy("event_year") \
    .agg(
        F.count("*").alias("shows"),
        F.sum("attendance").alias("total_attendance"),
        F.round(F.avg("attendance"), 0).alias("avg_attendance")
    ) \
    .orderBy("event_year") \
    .show()

print("\n🏟️ Exemplos de Shows Gerados:")
df_with_attendance.select(
    "event_date", "venue_name", "venue_city", "country", "attendance"
).orderBy(F.desc("attendance")).show(10, truncate=False)

# Atualizar df_unified com os dados de attendance
df_unified = df_with_attendance

print("\n✅ Dados de público gerados com sucesso!")

🎭 Gerando dados de público baseados em fatores realistas...

📊 Estatísticas de Público:

Total de shows: 129
Shows com público: 129
Shows sem público: 0

🎯 Público por País (Top 10):
+-------------+-----+----------------+--------------+
|country      |shows|total_attendance|avg_attendance|
+-------------+-----+----------------+--------------+
|Japan        |32   |1268700         |39647.0       |
|South Korea  |24   |769700          |32071.0       |
|United States|29   |624300          |21528.0       |
|Indonesia    |3    |159400          |53133.0       |
|Taiwan       |3    |119300          |39767.0       |
|Australia    |6    |95700           |15950.0       |
|China        |4    |60000           |15000.0       |
|Philippines  |3    |48800           |16267.0       |
|Hong Kong    |1    |46300           |46300.0       |
|Brazil       |3    |45000           |15000.0       |
+-------------+-----+----------------+--------------+
only showing top 10 rows

📅 Público por Ano:
+----------+----

In [0]:
# Salvar na camada gold
df_unified.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("skztour.03_gold.shows_unified")

print("✅ Tabela shows_unified salva com sucesso!")
print(f"Total de shows: {df_unified.count()}")
print(f"\nCaminho: skztour.03_gold.shows_unified")
print("\n🚀 Próximos passos:")
print("  1. Criar análises agregadas (shows por país, por ano, etc)")
print("  2. Adicionar continent (derivado de country)")
print("  3. Criar dimensões para Star Schema (opcional)")
print("  4. Criar dashboards de visualização")

✅ Tabela shows_unified salva com sucesso!
Total de shows: 129

Caminho: skztour.03_gold.shows_unified

🚀 Próximos passos:
  1. Criar análises agregadas (shows por país, por ano, etc)
  2. Adicionar continent (derivado de country)
  3. Criar dimensões para Star Schema (opcional)
  4. Criar dashboards de visualização
